# MarketPulse — Phase 4: QLoRA fine-tune of Phi-3-mini for market regime classification

**Run this on Google Colab with a T4 GPU runtime (free tier).**

## What this notebook does

Fine-tunes Microsoft's [Phi-3-mini-4k-instruct](https://huggingface.co/microsoft/Phi-3-mini-4k-instruct) (3.8B params) to classify market regimes as **bull / bear / sideways** from a tiny numerical snapshot. We use QLoRA — 4-bit quantization for the base model + a small trainable LoRA adapter — so the whole job fits in a free T4's 16 GB.

Outputs:
1. A LoRA adapter checkpoint, merged + pushed to your HuggingFace Hub at `<your-username>/phi3-regime-classifier`.
2. The critic service in MarketPulse will then call this model via HF Inference API at runtime — no GPU needed in production.

## Steps

1. Install deps (Colab-only — these are NOT in the project's pyproject.toml).
2. Build synthetic regime labels from historical OHLCV (forward-return labeling).
3. Format prompts for instruction-style training.
4. Load Phi-3-mini in 4-bit, attach a LoRA adapter via PEFT.
5. Train with `transformers.Trainer` (~30-60 min on T4).
6. Merge the adapter back into the base and push to Hub.
7. Smoke test: classify one fresh ticker.

See `learning/12-qlora-finetuning-phi3.md` for *why* every step looks the way it does.

## 1. Setup

Switch the Colab runtime to T4 GPU: **Runtime → Change runtime type → T4 GPU**. Then run this cell.

In [ ]:
!nvidia-smi  # confirm we have a T4 (~16GB VRAM)

In [ ]:
# bitsandbytes provides 4-bit quantization (CUDA-only — Colab T4 is CUDA, ✓).
# peft provides LoRA adapter wrappers.
# accelerate is needed for distributed training (single-GPU here but still required).
# datasets gives us a clean train/eval loop for transformers.Trainer.
!pip install -q -U \
    transformers==4.45.2 \
    accelerate==1.0.1 \
    peft==0.13.2 \
    bitsandbytes==0.44.1 \
    datasets==3.0.2 \
    yfinance==0.2.40 \
    huggingface_hub==0.26.0

In [ ]:
# Authenticate to HuggingFace Hub so we can push the fine-tuned model.
# Run this once. It opens a prompt asking for your HF token (with WRITE scope).
# Create one at https://huggingface.co/settings/tokens (set scope: 'write').
from huggingface_hub import login
login()

## 2. Build the dataset

**The training data is synthetic-supervised.** We download a few years of OHLCV via yFinance and label each row based on the **forward 20-day return** at that point in time:

- `bull` if the next 20 days returned ≥ +5%
- `bear` if the next 20 days returned ≤ −5%
- `sideways` otherwise

We then summarize each row as a small *snapshot* — the same five fields the runtime classifier sees — and pair it with its label.

**Important:** the label uses the *future* (forward returns), but at INFERENCE the model only sees the *past* snapshot. So training is supervised by the future; inference operates on what's known. This is fine; just don't accidentally include any forward-looking features in the snapshot (we don't).

See `learning/13-synthetic-supervised-labels.md` for the deeper why.

In [ ]:
import numpy as np
import pandas as pd
import yfinance as yf

# Tickers — diversify across mega-cap stocks + index + crypto + an ETF
# so the classifier sees regimes it'll encounter at inference time.
TICKERS = ['AAPL', 'MSFT', 'NVDA', 'GOOGL', 'TSLA', 'SPY', 'BTC-USD', 'ETH-USD']

def compute_rsi(close: pd.Series, period: int = 14) -> pd.Series:
    """Wilder RSI — matches the formula used in services/data_ingest."""
    delta = close.diff()
    gain = delta.where(delta > 0, 0.0)
    loss = -delta.where(delta < 0, 0.0)
    avg_gain = gain.ewm(alpha=1/period, adjust=False, min_periods=period).mean()
    avg_loss = loss.ewm(alpha=1/period, adjust=False, min_periods=period).mean()
    rs = avg_gain / avg_loss.replace(0, np.nan)
    return (100 - 100 / (1 + rs)).fillna(50.0)

def label_regime(forward_return: float, bull_thresh: float = 0.05, bear_thresh: float = -0.05) -> str:
    if forward_return >= bull_thresh:
        return 'bull'
    if forward_return <= bear_thresh:
        return 'bear'
    return 'sideways'

def build_ticker_dataset(ticker: str, period: str = '5y') -> pd.DataFrame:
    """For one ticker, build a frame of (snapshot, label) rows.
    
    Each row's label is the regime the market RESOLVED to over the next 20 days.
    At inference the classifier sees only the past 20 days; labels come from the future.
    """
    raw = yf.Ticker(ticker).history(period=period, interval='1d', auto_adjust=False)
    if raw.empty:
        return pd.DataFrame()
    
    df = pd.DataFrame(index=raw.index)
    df['close'] = raw['Close']
    df['return_5d'] = df['close'].pct_change(5)
    df['return_20d'] = df['close'].pct_change(20)
    df['vol_20d'] = df['close'].pct_change().rolling(20).std()
    df['rsi_14'] = compute_rsi(df['close'])
    # Forward 20-day return = LABEL signal. shift(-20) moves the future to the present row.
    df['forward_return_20d'] = df['close'].pct_change(20).shift(-20)
    df['ticker'] = ticker
    df['sentiment_24h'] = 0.0  # Phase 4 trains on no-news signal; Phase 5 adds it
    
    # Drop rows missing any feature or the forward label.
    df = df.dropna().copy()
    df['label'] = df['forward_return_20d'].apply(label_regime)
    return df

frames = [build_ticker_dataset(tk) for tk in TICKERS]
df_all = pd.concat([f for f in frames if not f.empty])
print(f'Total rows: {len(df_all):,}')
print(df_all['label'].value_counts())

In [ ]:
# Balance the dataset so the model doesn't just learn 'sideways is most common'.
# Undersample the majority classes to the minority size.
min_count = df_all['label'].value_counts().min()
balanced = pd.concat([
    df_all[df_all['label'] == lbl].sample(n=min_count, random_state=42)
    for lbl in ['bull', 'bear', 'sideways']
])
print(f'Balanced rows: {len(balanced):,}  (per class: {min_count})')
balanced = balanced.sample(frac=1, random_state=42).reset_index(drop=True)

In [ ]:
# Convert each row to the instruction-style prompt format Phi-3 expects.
# This prompt MUST match the one in services/critic/src/critic/classifier/phi3_regime.py:PROMPT_TEMPLATE.
PROMPT = (
    "Classify the market regime for the following snapshot as one of: "
    "bull, bear, sideways.\n\n"
    "Ticker: {ticker}\n"
    "5-day return: {return_5d:+.2%}\n"
    "20-day return: {return_20d:+.2%}\n"
    "20-day volatility: {vol_20d:.4f}\n"
    "RSI(14): {rsi_14:.1f}\n"
    "Sentiment(24h): {sentiment_24h:.2f}\n\n"
    "Answer with one word: bull, bear, or sideways.\n"
    "Answer: "
)

def format_row(row) -> dict:
    prompt = PROMPT.format(
        ticker=row['ticker'], return_5d=row['return_5d'], return_20d=row['return_20d'],
        vol_20d=row['vol_20d'], rsi_14=row['rsi_14'], sentiment_24h=row['sentiment_24h'],
    )
    return {'text': prompt + row['label']}

from datasets import Dataset
ds = Dataset.from_list([format_row(r) for _, r in balanced.iterrows()])
ds = ds.train_test_split(test_size=0.1, seed=42)
print(ds)

## 3. Load Phi-3 in 4-bit + attach LoRA adapter

QLoRA in one paragraph: the base model's weights are stored in 4-bit (saves ~75% of VRAM). A small set of *adapter* weights — the rank-`r` low-rank matrices inserted at every attention layer — are trained in fp16. We only ever touch ~0.1% of the parameter count, but it works because most fine-tuning gains come from a low-rank subspace of weight updates. See `learning/12-qlora-finetuning-phi3.md` for the longer story.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

BASE_MODEL = 'microsoft/Phi-3-mini-4k-instruct'

# 4-bit quant config — NF4 is the best general-purpose 4-bit format from the QLoRA paper.
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,  # quantize the quantization constants too — saves ~0.4 bits/param
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token  # Phi-3 doesn't ship with a pad token

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
)
model = prepare_model_for_kbit_training(model)

# LoRA hyperparameters — r=16 is the standard "works for most tasks" choice.
# We target ALL linear projections in attention; matching the QLoRA paper.
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()  # should report ~0.1% trainable

## 4. Train

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

def tokenize(batch):
    out = tokenizer(batch['text'], truncation=True, max_length=256, padding=False)
    return out

ds_tok = ds.map(tokenize, batched=True, remove_columns=['text'])

training_args = TrainingArguments(
    output_dir='./phi3-regime-out',
    num_train_epochs=2,                  # 2 epochs is usually enough — watch eval_loss for overfit
    per_device_train_batch_size=4,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=4,       # effective batch size = 4 × 4 = 16
    learning_rate=2e-4,                  # LoRA wants higher LR than full-model fine-tune
    warmup_ratio=0.03,
    lr_scheduler_type='cosine',
    logging_steps=20,
    eval_strategy='steps',
    eval_steps=200,
    save_strategy='steps',
    save_steps=200,
    save_total_limit=2,                  # don't fill the disk with checkpoints
    bf16=True,                            # T4 supports bf16; faster than fp32, more stable than fp16
    report_to='none',
    push_to_hub=False,                    # we push manually after merging
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=ds_tok['train'],
    eval_dataset=ds_tok['test'],
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False),
)

trainer.train()

## 5. Merge adapter + push to Hub

We merge the LoRA adapter back into the base model and upload the **merged** weights to your Hub. This means the HF Inference API can serve the model directly without needing to know about PEFT.

Replace `your-username` with your actual HF username.

In [ ]:
HF_USERNAME = 'your-username'  # ← CHANGE THIS
MODEL_REPO = f'{HF_USERNAME}/phi3-regime-classifier'

# Merge LoRA adapter into base weights so the merged checkpoint is a plain causal LM
# (no PEFT inference dependency at serving time).
merged = model.merge_and_unload()

merged.push_to_hub(MODEL_REPO, private=False)
tokenizer.push_to_hub(MODEL_REPO)

print(f'Pushed to https://huggingface.co/{MODEL_REPO}')
print(f'\nSet HF_REGIME_MODEL_ID={MODEL_REPO} in your .env to use it from the critic service.')

## 6. Smoke test against HF Inference API

Same code path the critic service uses in production. If this returns a sensible regime, you're done.

In [ ]:
from huggingface_hub import InferenceClient
import os

client = InferenceClient(token=os.environ.get('HF_TOKEN') or None)

sample = {
    'ticker': 'AAPL',
    'return_5d': 0.025,
    'return_20d': 0.08,
    'vol_20d': 0.018,
    'rsi_14': 62.0,
    'sentiment_24h': 0.0,
}

prompt = PROMPT.format(**sample)
out = client.text_generation(
    prompt=prompt,
    model=MODEL_REPO,
    max_new_tokens=10,
    temperature=0.1,
    do_sample=False,
)
print('Output:', out)

## Next steps

1. **Set the env var** in your local `.env`:
   ```
   HF_TOKEN=hf_yourtoken_with_read_scope
   HF_REGIME_MODEL_ID=your-username/phi3-regime-classifier
   ```
2. **Restart the critic service** so it picks up the new env vars. The `/critique` endpoint will now run the Phi-3 classifier when the orchestrator sends a `snapshot` field.
3. **Phase 5** will use this regime as an INPUT to the hybrid-RAG retrieval (we'll search for historical analogues whose Phi-3-classified regime matches the current one).

## Tips & troubleshooting

- **HF Inference API cold start.** First call after idle ~5-30s. Subsequent calls ~1s. Acceptable for our hourly cron.
- **Eval loss not dropping.** Sanity check: print a few `ds_tok['train'][0]` entries. The prompt should end with one of `bull|bear|sideways`.
- **CUDA OOM.** Lower `per_device_train_batch_size` to 2 and bump `gradient_accumulation_steps` to 8 to keep effective batch size constant.
- **Push fails with 403.** Your HF token needs `write` scope. Regenerate at https://huggingface.co/settings/tokens.
- **Want a faster smoke run?** Comment out 6 of the 8 tickers in the `TICKERS` list and set `num_train_epochs=1`.